# Bước 1: Chuẩn Hóa Dữ Liệu Multi-Scale & Xuất Tập Huấn Luyện (Data Scaling & Normalization)
Notebook này xử lý **chuẩn hóa Z-Score và Min-Max Scaling** cho toàn bộ 4 quy mô dung lượng MIMIC-III:
1. **Mức 1,360 mẫu** ➔ `mimic_zscore_1360.csv` & `mimic_minmax_1360.csv`
2. **Mức 4,083 mẫu** ➔ `mimic_zscore_4083.csv` & `mimic_minmax_4083.csv` (Mặc định: `mimic_zscore_scaled.csv` & `mimic_minmax_scaled.csv`)
3. **Mức 8,165 mẫu** ➔ `mimic_zscore_8165.csv` & `mimic_minmax_8165.csv`
4. **Mức 16,358 mẫu** ➔ `mimic_zscore_16358.csv` & `mimic_minmax_16358.csv`

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import os
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (16, 10)

### 1. Nạp tất cả các file đặc trưng multi-scale từ `data/features/`

In [2]:
data_dir_candidates = ['../../data/features', '../data/features', 'data/features']
data_dir = next((d for d in data_dir_candidates if os.path.exists(d)), None)
if not data_dir:
    raise FileNotFoundError("❌ Không tìm thấy thư mục data/features!")

tags = ['1360', '4083', '8165', '16358']
datasets = {}

for tag in tags:
    f_path = os.path.join(data_dir, f'mimic_features_{tag}.csv')
    if os.path.exists(f_path):
        df = pd.read_csv(f_path)
        datasets[tag] = df
        print(f"✅ Đã nạp mimic_features_{tag}.csv: {df.shape[0]} mẫu, {df.shape[1]} cột")

if not datasets:
    # Fallback default file
    def_path = os.path.join(data_dir, 'mimic_features.csv')
    if os.path.exists(def_path):
        df = pd.read_csv(def_path)
        datasets['4083'] = df
        print(f"✅ Đã nạp mimic_features.csv: {df.shape[0]} mẫu")

✅ Đã nạp mimic_features_1360.csv: 1360 mẫu, 17 cột
✅ Đã nạp mimic_features_4083.csv: 4083 mẫu, 17 cột
✅ Đã nạp mimic_features_8165.csv: 8165 mẫu, 17 cột
✅ Đã nạp mimic_features_16358.csv: 16358 mẫu, 17 cột


### 2. Thực hiện Chuẩn hóa Z-Score & Min-Max Scaling cho cả 4 quy mô

In [3]:
scaled_results = {}

for tag, df in datasets.items():
    X = df.drop(columns=['status'])
    y = df['status']
    feature_cols = X.columns.tolist()
    
    # Z-Score Normalization
    scaler_z = StandardScaler()
    X_z = pd.DataFrame(scaler_z.fit_transform(X), columns=feature_cols)
    df_zscore = pd.concat([X_z, y.reset_index(drop=True)], axis=1)
    
    # Min-Max Scaling
    scaler_mm = MinMaxScaler()
    X_mm = pd.DataFrame(scaler_mm.fit_transform(X), columns=feature_cols)
    df_minmax = pd.concat([X_mm, y.reset_index(drop=True)], axis=1)
    
    scaled_results[tag] = {
        'zscore': df_zscore,
        'minmax': df_minmax
    }
    print(f"⚡ Chuẩn hóa Z-Score & Min-Max thành công cho quy mô {tag} mẫu!")

⚡ Chuẩn hóa Z-Score & Min-Max thành công cho quy mô 1360 mẫu!
⚡ Chuẩn hóa Z-Score & Min-Max thành công cho quy mô 4083 mẫu!
⚡ Chuẩn hóa Z-Score & Min-Max thành công cho quy mô 8165 mẫu!
⚡ Chuẩn hóa Z-Score & Min-Max thành công cho quy mô 16358 mẫu!


### 3. Xuất tất cả các file đã scaled vào `data/processed/`

In [4]:
output_dir_candidates = ['../../data/processed', '../data/processed', 'data/processed']
output_dir = next((d for d in output_dir_candidates if os.path.exists(os.path.dirname(d))), '../../data/processed')
os.makedirs(output_dir, exist_ok=True)

for tag, res in scaled_results.items():
    z_path = os.path.join(output_dir, f'mimic_zscore_{tag}.csv')
    mm_path = os.path.join(output_dir, f'mimic_minmax_{tag}.csv')
    
    res['zscore'].to_csv(z_path, index=False)
    res['minmax'].to_csv(mm_path, index=False)
    
    print(f"🎉 [{tag} Mẫu] Z-Score -> {z_path}")
    print(f"🎉 [{tag} Mẫu] Min-Max -> {mm_path}")
    
    if tag == '4083':
        res['zscore'].to_csv(os.path.join(output_dir, 'mimic_zscore_scaled.csv'), index=False)
        res['minmax'].to_csv(os.path.join(output_dir, 'mimic_minmax_scaled.csv'), index=False)
        print(f"⭐ Đã cập nhật file mặc định mimic_zscore_scaled.csv & mimic_minmax_scaled.csv")

🎉 [1360 Mẫu] Z-Score -> ../../data/processed\mimic_zscore_1360.csv
🎉 [1360 Mẫu] Min-Max -> ../../data/processed\mimic_minmax_1360.csv
🎉 [4083 Mẫu] Z-Score -> ../../data/processed\mimic_zscore_4083.csv
🎉 [4083 Mẫu] Min-Max -> ../../data/processed\mimic_minmax_4083.csv
⭐ Đã cập nhật file mặc định mimic_zscore_scaled.csv & mimic_minmax_scaled.csv
🎉 [8165 Mẫu] Z-Score -> ../../data/processed\mimic_zscore_8165.csv
🎉 [8165 Mẫu] Min-Max -> ../../data/processed\mimic_minmax_8165.csv
🎉 [16358 Mẫu] Z-Score -> ../../data/processed\mimic_zscore_16358.csv
🎉 [16358 Mẫu] Min-Max -> ../../data/processed\mimic_minmax_16358.csv
